# Iniciar Madrid Pádel en JupyterHub
Borja Ibáñez Pérez · ICAI

Este notebook inicia la aplicación en el JupyterHub de ICAI. Los cálculos están en `analisis.py`, la interfaz en `app.py` y el diseño en `assets/estilos.css`.

Colócalo junto a esos archivos. El CSV debe estar en `Datos/app/reservas_padel_app.csv`. No necesitas volver a ejecutar el notebook 02 cada vez que abres la aplicación.


## 1. Instalar las dependencias
Ejecuta esta celda la primera vez o cuando falten dependencias. Si reinicias el servidor y se pierden las instalaciones del entorno, vuelve a ejecutarla. Si se solicita reiniciar el kernel, hazlo antes de continuar.


In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 2. Comprobar archivos

In [2]:
from pathlib import Path
for archivo in ["app.py", "analisis.py", "assets/estilos.css", "Datos/app/reservas_padel_app.csv"]:
    assert Path(archivo).is_file(), f"Falta el archivo: {archivo}. Revisa la carpeta del notebook."
print("Archivos preparados.")

Archivos preparados.


## 3. Comprobar el componente proxy
Esta celda comprueba que el paquete puede importarse desde el kernel. No confirma por sí sola que la extensión esté activa en el servidor Jupyter.


In [3]:
import jupyter_server_proxy
print("Proxy disponible:", jupyter_server_proxy.__file__)

Proxy disponible: /home/202207424/.local/lib/python3.13/site-packages/jupyter_server_proxy/__init__.py


## 4. Iniciar la aplicación

In [4]:
import os
import sys
import time
import subprocess
from pathlib import Path
from urllib.request import urlopen
from IPython.display import display, Markdown

carpeta = Path.cwd()

if not (carpeta / "app.py").is_file():
    raise FileNotFoundError(
        "No se encuentra app.py. Abre este notebook dentro de TFG analytics."
    )

prefijo = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")

if not prefijo:
    raise RuntimeError("No se ha encontrado la ruta de JupyterHub.")

puerto = 8060
ruta_proxy = prefijo.rstrip("/") + f"/proxy/{puerto}/"
enlace = "https://jupyterhubicai.comillas.edu" + ruta_proxy

entorno = os.environ.copy()
entorno["HOST"] = "127.0.0.1"
entorno["PORT"] = str(puerto)
entorno["DASH_REQUESTS_PATHNAME_PREFIX"] = ruta_proxy
entorno["DASH_ROUTES_PATHNAME_PREFIX"] = "/"

if "proceso_dash" in globals() and proceso_dash.poll() is None:
    proceso_dash.terminate()
    proceso_dash.wait(timeout=10)

archivo_log = carpeta / "dash_inicio.log"

with archivo_log.open("w", encoding="utf-8") as registro:
    proceso_dash = subprocess.Popen(
        [sys.executable, "-u", "app.py"],
        cwd=carpeta,
        env=entorno,
        stdout=registro,
        stderr=subprocess.STDOUT
    )

print("Arrancando Madrid Pádel…", flush=True)

lista = False
limite = time.monotonic() + 30

while time.monotonic() < limite:
    if proceso_dash.poll() is not None:
        break

    try:
        with urlopen(
            f"http://127.0.0.1:{puerto}/_dash-layout",
            timeout=1
        ) as respuesta:
            lista = respuesta.status == 200
    except OSError:
        pass

    if lista:
        break

    time.sleep(0.5)

if lista and proceso_dash.poll() is None:
    print("Dash responde dentro del servidor.")
    display(Markdown(f"**[Abrir Madrid Pádel]({enlace})**"))
    print("Abre el enlace para comprobar el acceso desde el navegador.")
else:
    if proceso_dash.poll() is None:
        print("El proceso sigue arrancando, pero aún no responde.")
    else:
        print("El proceso de Dash se ha detenido.")

    print("\nÚltimos mensajes:")
    print(
        archivo_log.read_text(
            encoding="utf-8", errors="replace"
        )[-5000:]
    )

Arrancando Madrid Pádel…
Dash responde dentro del servidor.


**[Abrir Madrid Pádel](https://jupyterhubicai.comillas.edu/user/202207424/proxy/8060/)**

Abre el enlace para comprobar el acceso desde el navegador.
